CSV 파일 없이 관측 데이터를 직접 만들어 분석하는 과제입니다. 아래 순서대로 진행하세요.

① 랜덤 시계열 Series 생성

np.random.seed(2024)로 시드를 고정하세요.

pd.date_range로 2024년 3월 1일부터 90일간의 날짜 인덱스를 만드세요.

평균 18, 표준편차 4인 정규분포 난수(np.random.normal) 90개를 소수 첫째 자리까지 반올림해 값으로 사용하고, 위 날짜를 인덱스로 갖는 Series s를 만드세요.

head(), describe(), index, dtype을 출력해 구조를 확인하세요.

센서 고장 상황을 인위적으로 만듭니다. np.random.choice로 인덱스 10개를 중복 없이 뽑아 그 위치의 값을 결측값(np.nan)으로 바꾸세요.

이상치도 주입합니다. 결측이 아닌 인덱스 중 5개를 중복 없이 뽑아 그 값을 3배로 만드세요.

② 결측값 보정 방식 4가지 비교

결측이 발생한 날짜 목록과 개수를 출력하세요.

다음 4가지 Series를 각각 만드세요. (1) 결측 제거 (2) 0으로 채움 (3) 전체 평균으로 채움 (4) 바로 앞 날짜 값으로 채움

네 결과의 평균과 표준편차를 하나의 DataFrame으로 정리해 비교하고, 시계열 데이터에서 어떤 방식이 가장 부적절한지 주석으로 근거와 함께 적으세요.

이후 단계에서는 (4) 앞값 채움 결과를 사용하세요.

③ 이상치 탐지와 보정

평균 ± (2 × 표준편차)를 상·하한으로 계산해 출력하세요.

조건 색인으로 상한을 넘거나 하한 밑인 날짜와 값을 출력하고, ①에서 주입한 5일과 일치하는지 확인하세요.

np.where를 중첩해 상한 초과는 상한값으로, 하한 미만은 하한값으로 바꾼 Series clean을 만드세요.

보정 전후의 describe()를 비교해 표준편차가 어떻게 변했는지 주석으로 적으세요.

④ 월별 집계와 이동평균

clean을 월 기준으로 그룹화해 관측일수·평균·최저·최고를 한 번에 구하세요. (groupby + agg)

7일 이동평균(rolling)을 구하고, 앞쪽 6개가 결측인 이유를 주석으로 적으세요.

기온 상위 5일을 정렬해 출력하세요.

전체 평균보다 높았던 날이 며칠인지 출력하세요.

In [4]:
# %pip install numpy pandas
import numpy as np
import pandas as pd

In [6]:
np.random.seed(2024)
date=pd.date_range(start="2024/03/01",periods=90)
s=pd.Series(np.random.normal(18,4,90).round(1),index=date)

In [7]:
print(s.head())
print(s.describe())
print(s.index)
print(s.dtype)

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
Freq: D, dtype: float64
count    90.000000
mean     18.202222
std       3.923996
min       7.500000
25%      15.800000
50%      18.300000
75%      21.350000
max      25.700000
dtype: float64
DatetimeIndex(['2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04',
               '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08',
               '2024-03-09', '2024-03-10', '2024-03-11', '2024-03-12',
               '2024-03-13', '2024-03-14', '2024-03-15', '2024-03-16',
               '2024-03-17', '2024-03-18', '2024-03-19', '2024-03-20',
               '2024-03-21', '2024-03-22', '2024-03-23', '2024-03-24',
               '2024-03-25', '2024-03-26', '2024-03-27', '2024-03-28',
               '2024-03-29', '2024-03-30', '2024-03-31', '2024-04-01',
               '2024-04-02', '2024-04-03', '2024-04-04', '2024-04-05',
               '2024-04-06', '2024-04-07', '2024-04-08', '2024-04-09',
    

In [8]:
none_idx=np.random.choice(s.index,size=10,replace=False)
s[none_idx]=np.nan
outlier_idx=np.random.choice(s.index,size=5,replace=False)
s[outlier_idx]=s[outlier_idx]*3

In [11]:
s_drop=s.dropna() #nan
s_zero=s.fillna(0) #zerofill
s_mean=s.fillna(s.mean()) #mean
s_forward=s.ffill() #forward fill

In [12]:
compare=pd.DataFrame(
    {
        'nan':[s_drop.mean(),s_drop.std()],
        'zerofill':[s_zero.mean(),s_zero.std()],
        'mean':[s_mean.mean(),s_mean.std()],
        'forwardfill':[s_forward.mean(),s_forward.std()]
    }, index=['mean','std']
)
print(compare.round(2))

        nan  zerofill   mean  forwardfill
mean  19.66     17.48  19.66        19.89
std    8.72     10.30   8.21         8.30


In [ ]:
# nan과 mean 방식은 측정치의 평균을 왜곡하지 않지만, nan에선 표본수가 줄어들면서 표준편차가 증가
# forwardfill 방식은 사실상 random 값을 대입하는 것과 유사
# zerofill 방식은 통계 분석에 실제 반영되는 0이라는 값으로 인해 통계치가 크게 왜곡됨

In [ ]:
# 이상치 탐지 및 보정

low=s_forward.mean()-2*s_forward.std()
high=s_forward.mean()+2*s_forward.std()
print('하한값:',round(low,2),'상한값:',round(high,2))

print(s_forward[(s_forward<low)|(s_forward>high)])

하한값: 3.29 상한값: 36.49
2024-03-19    65.4
2024-04-08    52.5
2024-05-22    57.0
dtype: float64


In [ ]:
clean=pd.Series(np.where(
    s_forward>high,high,np.where(
        s_forward<low,low,s_forward
        )
    ),index=s_forward.index)
print(s_forward.describe())
print(clean.describe()) # outlier가 범위 내로 좁혀지면서 분산이 줄어들었다.

count    90.000000
mean     19.893333
std       8.299544
min       7.500000
25%      16.075000
50%      18.750000
75%      21.975000
max      65.400000
dtype: float64
count    90.000000
mean     19.166414
std       5.191406
min       7.500000
25%      16.075000
50%      18.750000
75%      21.975000
max      36.492421
dtype: float64


In [ ]:
monthly=clean.groupby(clean.index.month).agg(['count','mean','min','max']).round(2)
print(monthly)
roll=clean.rolling(7).mean() # 앞쪽으로 7일이 모여야 하므로 모이지 않는 6일 간은 계산 불가
print(roll.head(10))

   count   mean   min    max
3     31  18.61   7.5  36.49
4     30  19.58  12.7  36.49
5     29  19.34   8.5  36.49
2024-03-01          NaN
2024-03-02          NaN
2024-03-03          NaN
2024-03-04          NaN
2024-03-05          NaN
2024-03-06          NaN
2024-03-07    18.857143
2024-03-08    17.142857
2024-03-09    16.985714
2024-03-10    17.157143
Freq: D, dtype: float64


In [21]:
print(clean.sort_values(ascending=False).head(5))
print('평균 초과 일수:',len(clean[clean>clean.mean()]))

2024-03-19    36.492421
2024-04-08    36.492421
2024-05-22    36.492421
2024-05-11    25.700000
2024-05-12    25.700000
dtype: float64
평균 초과 일수: 44


가상의 쇼핑몰 주문 데이터를 직접 만들고, 실제 정산 로직을 적용하는 과제입니다.

① 랜덤 주문 데이터 생성

np.random.seed(7), 주문 건수 300건으로 설정하세요.

아래 규칙으로 열을 만들어 DataFrame df를 생성하고, 주문번호를 인덱스로 지정하세요.

주문번호: ORD0001 ~ ORD0300 (4자리 0채움)

주문일: 2024-01-01 기준 0~179일 사이 랜덤 경과일

회원등급: 일반 50%, 실버 25%, 골드 15%, VIP 10% 확률로 추출

카테고리: 식품·의류·가전·도서·뷰티 중 랜덤

수량: 1~5 사이 랜덤 정수

단가: 5000·12000·25000·48000·99000 중 랜덤

쿠폰사용: True 30%, False 70%

head(), info(), describe()를 출력하고, 회원등급별 건수를 세어 의도한 비율과 비슷한지 확인하세요.

입력 누락 상황을 만듭니다. 랜덤한 20건의 수량을 결측값으로 바꾼 뒤, 결측 개수를 확인하고 중앙값으로 채우세요.

② 정산 로직을 파생 열로 구현

주문금액 = 수량 × 단가

등급할인율 = 일반 0%, 실버 3%, 골드 5%, VIP 10% (딕셔너리 + map 사용)

쿠폰할인율 = 쿠폰 사용 시 5%, 아니면 0% (np.where 사용)

총할인율 = 등급할인율 + 쿠폰할인율

할인금액 = 주문금액 × 총할인율 ÷ 100 (반올림)

배송비 = (주문금액 − 할인금액)이 30,000원 이상이면 0원, 아니면 3,000원

최종결제금액 = 주문금액 − 할인금액 + 배송비

주문규모 = 최종결제금액이 20만 이상 '대형', 5만 이상 '중형', 그 외 '소형' (사용자 정의 함수 + apply 사용)

③ 집계와 조건 분석

주문일에서 월을 뽑아 월 열을 추가하세요.

월별 주문 건수·매출 합계·평균 주문금액을 한 번에 구하세요.

카테고리별 × 회원등급별 매출 합계를 구하세요.

VIP 회원이면서 최종결제금액 10만 원 이상인 주문 건수를 구하세요.

쿠폰을 사용한 가전 카테고리 주문만 조회하세요.

카테고리별 매출 상위 2건씩을 뽑아 출력하세요.

④ 정리와 저장

중간 계산용 열 등급할인율, 쿠폰할인율을 drop으로 한 번에 삭제하세요.

결과를 orders.csv로 저장하세요. (엑셀에서 한글이 깨지지 않도록)

저장한 파일을 다시 읽되 주문번호를 인덱스로, 주문일을 날짜형으로 읽어오세요.

loc['ORD0010']과 iloc[9] 결과를 비교하고, 두 값이 같은 이유를 주석으로 적으세요.

In [22]:
import numpy as np
import pandas as pd

In [49]:
np.random.seed(7)
n=300
order_id=['ORD'+str(i) for i in range(1,n+1)] # 0 채움?
order_date=[pd.to_datetime('2024-01-01')+pd.DateOffset(i) for i in np.random.randint(0,180,n)]
level=np.random.choice(['일반','실버','골드','VIP'],n,p=[0.5,0.25,0.15,0.1])
category=np.random.choice(['식품','의류','가전','도서','뷰티'],n)
quantity=np.random.randint(1,6,n)
price=np.random.choice([5000,12000,25000,48000,99000],n)
coupon=np.random.choice([True,False],n,p=[0.3,0.7])

DF=pd.DataFrame({
    '주문번호':order_id,'주문일':order_date,
    '회원등급':level,'카테고리':category,
    '수량':quantity,'가격':price,
    '단가':price,'쿠폰사용':coupon
})
DF=DF.set_index('주문번호')

In [50]:
print(DF.head())
print(DF.info())
print(DF.describe())
print(DF['회원등급'].value_counts())

            주문일 회원등급 카테고리  수량     가격     단가   쿠폰사용
주문번호                                              
ORD1 2024-06-24   일반   도서   4  12000  12000   True
ORD2 2024-01-26   일반   식품   3   5000   5000  False
ORD3 2024-03-08   일반   식품   1  12000  12000   True
ORD4 2024-05-31   실버   의류   1  12000  12000   True
ORD5 2024-04-13   일반   가전   4  12000  12000  False
<class 'pandas.DataFrame'>
Index: 300 entries, ORD1 to ORD300
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   주문일     300 non-null    datetime64[us]
 1   회원등급    300 non-null    str           
 2   카테고리    300 non-null    str           
 3   수량      300 non-null    int32         
 4   가격      300 non-null    int64         
 5   단가      300 non-null    int64         
 6   쿠폰사용    300 non-null    bool          
dtypes: bool(1), datetime64[us](1), int32(1), int64(2), str(2)
memory usage: 15.5+ KB
None
                       주문일          수량            가격          

In [51]:
DF['수량']=DF['수량'].astype(float)
blank=np.random.choice(DF.index,20,replace=False)
DF.loc[blank,'수량']=np.nan

print('결측:',DF['수량'].isnull().sum())
DF['수량']=DF['수량'].fillna(DF['수량'].median())
print('결측:',DF['수량'].isnull().sum())

결측: 20
결측: 0


In [52]:
DF['주문금액']=DF['수량']*DF['단가']
rate={'일반':0,'실버':3,'골드':5,'VIP':10}
DF['등급할인율'] = DF['회원등급'].map(rate)
DF['쿠폰할인율'] = np.where(DF['쿠폰사용'], 5, 0)
DF['총할인율']   = DF['등급할인율'] + DF['쿠폰할인율']
DF['할인금액']   = (DF['주문금액'] * DF['총할인율'] / 100).round(0)
DF['배송비']     = np.where(DF['주문금액'] - DF['할인금액'] >= 30000, 0, 3000)
DF['최종결제금액'] = DF['주문금액'] - DF['할인금액'] + DF['배송비']

def size_label(i):
    if i>=200000:
        return '대형'
    elif i>=50000:
        return '중형'
    else:
        return '소형'

DF['주문규모']=DF['최종결제금액'].apply(size_label)

print(DF.head())
print(DF['주문규모'].value_counts())

            주문일 회원등급 카테고리   수량     가격     단가   쿠폰사용     주문금액  등급할인율  쿠폰할인율  \
주문번호                                                                         
ORD1 2024-06-24   일반   도서  4.0  12000  12000   True  48000.0      0      5   
ORD2 2024-01-26   일반   식품  3.0   5000   5000  False  15000.0      0      0   
ORD3 2024-03-08   일반   식품  1.0  12000  12000   True  12000.0      0      5   
ORD4 2024-05-31   실버   의류  1.0  12000  12000   True  12000.0      3      5   
ORD5 2024-04-13   일반   가전  4.0  12000  12000  False  48000.0      0      0   

      총할인율    할인금액   배송비   최종결제금액 주문규모  
주문번호                                    
ORD1     5  2400.0     0  45600.0   소형  
ORD2     0     0.0  3000  18000.0   소형  
ORD3     5   600.0  3000  14400.0   소형  
ORD4     8   960.0  3000  14040.0   소형  
ORD5     0     0.0     0  48000.0   소형  
주문규모
소형    152
중형    109
대형     39
Name: count, dtype: int64


In [53]:
DF['월']=DF['주문일'].dt.month
DF.groupby('월')['최종결제금액'].agg(['count','sum','mean']).round(0)
DF.groupby(['카테고리','회원등급'])['최종결제금액'].sum()
print(len(DF[(DF['회원등급'] == 'VIP') & (DF['최종결제금액'] >= 100000)]))
print(DF[DF['쿠폰사용'] & (DF['카테고리'] == '가전')])

top2 = DF.sort_values(by=['카테고리', '최종결제금액'], ascending=[True, False]) \
         .groupby('카테고리').head(2)
print(top2[['카테고리', '최종결제금액']])

12
              주문일 회원등급 카테고리   수량     가격     단가  쿠폰사용      주문금액  등급할인율  쿠폰할인율  \
주문번호                                                                           
ORD13  2024-05-16   일반   가전  5.0  48000  48000  True  240000.0      0      5   
ORD15  2024-03-09   일반   가전  4.0  48000  48000  True  192000.0      0      5   
ORD41  2024-04-28   일반   가전  4.0  48000  48000  True  192000.0      0      5   
ORD55  2024-04-10  VIP   가전  4.0  48000  48000  True  192000.0     10      5   
ORD73  2024-01-22   일반   가전  4.0  12000  12000  True   48000.0      0      5   
ORD86  2024-05-04   일반   가전  1.0   5000   5000  True    5000.0      0      5   
ORD101 2024-05-20   골드   가전  1.0  25000  25000  True   25000.0      5      5   
ORD129 2024-06-24   일반   가전  1.0  12000  12000  True   12000.0      0      5   
ORD131 2024-03-28   일반   가전  1.0  99000  99000  True   99000.0      0      5   
ORD146 2024-04-19  VIP   가전  3.0  25000  25000  True   75000.0     10      5   
ORD159 2024-03-18  VIP   가전  4.0  250

In [54]:
DF.drop(['등급할인율', '쿠폰할인율'], axis='columns', inplace=True)
DF.to_csv('orders.csv', encoding='UTF-8')

DF_2 = pd.read_csv('orders.csv', index_col='주문번호', header=0,
                  sep=',', parse_dates=['주문일'])
print(DF_2.dtypes)
print(DF_2.loc['ORD10'])
print(DF_2.iloc[9])

주문일       datetime64[us]
회원등급                 str
카테고리                 str
수량               float64
가격                 int64
단가                 int64
쿠폰사용                bool
주문금액             float64
총할인율               int64
할인금액             float64
배송비                int64
최종결제금액           float64
주문규모                 str
월                  int64
dtype: object
주문일       2024-03-30 00:00:00
회원등급                       실버
카테고리                       도서
수량                        4.0
가격                      48000
단가                      48000
쿠폰사용                    False
주문금액                 192000.0
총할인율                        3
할인금액                   5760.0
배송비                         0
최종결제금액               186240.0
주문규모                       중형
월                           3
Name: ORD10, dtype: object
주문일       2024-03-30 00:00:00
회원등급                       실버
카테고리                       도서
수량                        4.0
가격                      48000
단가                      48000
쿠폰사용     

분기별로 따로 만들어진 데이터를 하나로 합쳐 분석하는, 실무에 가장 가까운 형태의 과제입니다.

① 분기 데이터 생성 함수 만들고 연결하기

np.random.seed(99)로 고정하세요.

지점 5곳(서울·부산·대구·광주·대전) × 상품 3종(A·B·C)의 조합을 pd.MultiIndex.from_product로 만들고, 인덱스명을 지점, 상품으로 지정하세요.

분기명을 인자로 받아 아래 열을 가진 DataFrame을 반환하는 함수 make_quarter(q) 를 작성하세요.

판매량: 50~299 랜덤 정수 / 단가: 10000·15000·20000 중 랜덤 / 반품: 0~19 랜덤 정수 / 분기: 인자로 받은 값

이 함수로 1Q~4Q 데이터를 만들어 pd.concat으로 세로로 연결한 sales를 만드세요. (총 60행)

shape, index.names, head()로 구조를 확인하세요.

집계 누락 상황을 만듭니다. 랜덤한 8개 행의 판매량을 결측값으로 바꾼 뒤, 지점별 중앙값으로 채우세요. (groupby + transform)

② 파생 열과 MultiIndex 집계

매출 = 판매량 × 단가, 반품율 = 반품 ÷ 판매량 × 100 (소수 둘째 자리)

등급 열: 매출 400만 이상 'A', 200만 이상 'B', 그 외 'C' (np.where 중첩)

지점 기준으로 매출의 건수·합계·평균·최댓값을 한 번에 구하세요.

지점 × 상품 기준, 분기 × 지점 기준 매출 합계를 각각 구하세요.

등급별 행 개수를 구하세요.

③ 순위와 교차표(피벗)

지점별 연간 매출 합계를 내림차순 정렬하고, rank로 순위를 매기세요.

pivot_table로 행=지점, 열=분기, 값=매출 합계인 교차표 pv를 만드세요.

pv에 연간합계 열(행 방향 합계, axis=1)을 추가하고 내림차순 정렬하세요.

성장률 열 = (4Q − 1Q) ÷ 1Q × 100 (소수 첫째 자리)을 추가하고, 성장률 순으로 정렬해 출력하세요.

④ 가로 방향 연결과 저장

1Q 지점별 매출 합계와 4Q 지점별 매출 합계를 각각 Series로 만들고, pd.concat의 axis=1로 나란히 붙여 비교표를 만드세요. (keys로 열 이름 지정)

지점별 매출 1위 주문 행을 각각 1건씩 뽑아 출력하세요.

결과를 sales_report.csv로 저장한 뒤, 지점과 상품을 함께 인덱스로 지정해 다시 읽어오세요.

읽어온 데이터에서 loc['서울']로 서울 지점 데이터만 조회하고, 원본 sales와 인덱스 구조가 어떻게 달라졌는지 주석으로 적으세요.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(99)